In [2]:
# Import required libraries
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col, when, lit
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType,
)

print("✅ Imports successful!")
print(f"Python version: {sys.version}")

# Set environment variables for better output
os.environ["PYSPARK_SUBMIT_ARGS"] = "--jars /opt/spark/jars-custom/*.jar pyspark-shell"

✅ Imports successful!
Python version: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]


In [4]:
spark = (
    SparkSession.builder.appName("IcebergNessieDemo")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .config(
        "spark.sql.catalog.nessie.catalog-impl",
        "org.apache.iceberg.nessie.NessieCatalog",
    )
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1")
    .config("spark.sql.catalog.nessie.ref", "main")
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)

In [24]:
print("✅ SparkSession created successfully!")
print(f"Spark version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")

# Set default catalog
spark.sql("USE nessie")

✅ SparkSession created successfully!
Spark version: 3.5.0
Application ID: local-1751698590377


DataFrame[]

In [25]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.demo")

DataFrame[]

In [37]:
medals = spark.read.csv("s3a://sdc/medals.csv",header=True, inferSchema=True)
matches = spark.read.csv("s3a://sdc/matches.csv",header=True, inferSchema=True)
match_details = spark.read.csv("s3a://sdc/match_details.csv",header=True, inferSchema=True)
medals_matches_players = spark.read.csv("s3a://sdc/medals_matches_players.csv",header=True, inferSchema=True)

In [32]:
# match_details.printSchema()
match_details.write.bucketBy(16,"match_id").format("iceberg").mode("overwrite").saveAsTable("nessie.demo.match_details")

In [38]:
matches.write.bucketBy(16,"match_id").format("iceberg").mode("overwrite").saveAsTable("nessie.demo.matches")

In [40]:
medals.write.bucketBy(16,"medal_id").format("iceberg").mode("overwrite").saveAsTable("nessie.demo.medals")

In [41]:
medals_matches_players.write.bucketBy(16,"match_id").format("iceberg").mode("overwrite").saveAsTable("nessie.demo.medals_matches_players")

In [43]:
matches.join(match_details,on="match_id").join(medals_matches_players,on="match_id").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [match_id#2939, mapid#2940, is_team_game#2941, playlist_id#2942, game_variant_id#2943, is_match_over#2944, completion_date#2945, match_duration#2946, game_mode#2947, map_variant_id#2948, player_gamertag#2977, previous_spartan_rank#2978, spartan_rank#2979, previous_total_xp#2980, total_xp#2981, previous_csr_tier#2982, previous_csr_designation#2983, previous_csr#2984, previous_csr_percent_to_next_tier#2985, previous_csr_rank#2986, current_csr_tier#2987, current_csr_designation#2988, current_csr#2989, current_csr_percent_to_next_tier#2990, ... 24 more fields]
   +- SortMergeJoin [match_id#2939], [match_id#3065], Inner
      :- Sort [match_id#2939 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(match_id#2939, 200), ENSURE_REQUIREMENTS, [plan_id=774]
      :     +- Project [match_id#2939, mapid#2940, is_team_game#2941, playlist_id#2942, game_variant_id#2943, is_match_over#2944, completion_date#2945, match_du

In [50]:
import pyspark.sql.functions as F
matches = matches.withColumn("obs_year", F.year("completion_date")).withColumn("obs_month",F.month("completion_date"))
matches.write.format("iceberg").partitionBy("obs_year","obs_month").mode("overwrite").saveAsTable("nessie.demo.fact_matches")